# 03 — GFP & Component Statistics

Component peak latencies and analysis
windows are loaded from `outputs/component_peaks.json` (written by notebook 02)
instead of being re-detected here — ensuring both notebooks use identical definitions.

Run notebook 02 first (Setup cells 0–4) to generate or refresh the JSON.


## 0. Imports


In [ ]:
import sys, json, warnings
from pathlib import Path

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
import statsmodels.formula.api as smf
from erp_tools.evoked import compute_condition_gfp_grand_average

mne.set_log_level('WARNING')


## 1. Settings

Edit component search windows here. Each entry: `(name, polarity, tmin_ms, tmax_ms)`  
polarity `+1` = positive peak (argmax), `-1` = negative peak (argmin) on GFP.

In [ ]:
BIDS_ROOT = '/workspace/data/marmoset_FreqIntensity'

SUBJECTS = ['Cj399', 'Cj459']

# Reference label → (pipeline_name, gfp_exclude, amp_exclude)
# gfp_exclude : channels excluded from GFP (zero-valued ref channels only)
# amp_exclude : channels excluded from peak-channel reporting (same channels + any others)
REFERENCES = {
    'linkedEars': ('erp-pipeline-ref-linkedEars', [],       ['A1', 'A2']),
    'CAR':        ('erp-pipeline-ref-CAR',        [],       []),
    'CMR':        ('erp-pipeline-ref-CMR',        [],       []),
}

# Intensity and frequency levels to look for
INTENSITIES = ['45dB', '60dB', '75dB']
FREQUENCIES = ['00125Hz', '00250Hz', '00500Hz', '01000Hz',
               '02000Hz', '04000Hz', '08000Hz', '16000Hz']

LATENCY_INTENSITY = '75dB'   # intensity level used for peak-latency determination

# ── Component search windows (edit freely) ──────────────────────────────────
# (component_name, polarity, tmin_ms, tmax_ms)
COMPONENTS = [
    ('P1',  +1,  10,  20),
    ('N1',  -1,  30,  60),
    ('P2',  +1,  60,  100),
    ('N2',  -1, 120,  180),
    ('LN',  -1, 200,  600),   # late negativity
]

# ── Analysis window half-widths (ms) centred at peak latency ─────────────────
# Full window = peak_latency ± half-width
COMPONENT_WINDOW_MS = {
    'P1':  5 / 2,   #  5 ms wide
    'N1': 10 / 2,   # 10 ms wide
    'P2': 20 / 2,   # 20 ms wide
    'N2': 40 / 2,   # 40 ms wide
    'LN': 80 / 2,   # 80 ms wide
}
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR = REPO_ROOT / 'outputs' / 'stats'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Helper functions

In [ ]:
def get_all_eeg_picks(evoked: mne.Evoked, gfp_exclude: list = []) -> np.ndarray:
    """EEG picks for GFP — excludes zero-valued ref channels (empty for linkedEars, CAR, CMR)."""
    return mne.pick_types(evoked.info, eeg=True, exclude=gfp_exclude)


def get_signal_picks(evoked: mne.Evoked, amp_exclude: list) -> np.ndarray:
    """EEG channel indices excluding reference electrodes — used for peak_channel reporting."""
    return mne.pick_types(evoked.info, eeg=True, exclude=amp_exclude)


def compute_gfp(data_uv: np.ndarray) -> np.ndarray:
    """GFP (population std, ddof=0) across channels at each time point.
    data_uv shape: (n_ch, n_times).
    """
    return data_uv.std(axis=0, ddof=0)


def find_peak_on_gfp(
    evoked: mne.Evoked,
    gfp: np.ndarray,
    data_uv: np.ndarray,
    picks: np.ndarray,
    tmin_ms: float,
    tmax_ms: float,
) -> dict:
    """
    Find the GFP maximum within [tmin_ms, tmax_ms].
    Returns latency, GFP amplitude, and the signal channel with the largest
    absolute amplitude at that latency.

    data_uv / picks: signal channels only (reference excluded).
    """
    times_ms = evoked.times * 1000
    mask = (times_ms >= tmin_ms) & (times_ms <= tmax_ms)

    if not mask.any():
        return dict(latency_ms=np.nan, gfp_amplitude_uv=np.nan,
                    channel='', channel_amplitude_uv=np.nan)

    peak_idx = int(np.where(mask)[0][np.argmax(gfp[mask])])
    lat_ms   = float(times_ms[peak_idx])
    gfp_amp  = float(gfp[peak_idx])

    col     = data_uv[:, peak_idx]
    ch_idx  = int(np.argmax(np.abs(col)))
    ch_name = evoked.ch_names[picks[ch_idx]]
    ch_amp  = float(col[ch_idx])

    return dict(
        latency_ms=lat_ms,
        gfp_amplitude_uv=gfp_amp,
        channel=ch_name,
        channel_amplitude_uv=ch_amp,
    )


def load_evokeds_for_subject(
    bids_root: str,
    subject: str,
    pipeline: str,
    intensities: list,
    frequencies: list,
) -> dict:
    """Load all available condition evoked files for one subject.
    Returns {condition_label: mne.Evoked}, silently skipping missing files.
    """
    deriv = Path(bids_root) / 'derivatives' / pipeline / f'sub-{subject}' / 'eeg'
    evokeds = {}
    for intensity in intensities:
        task = f'Passive{intensity}'
        for freq in frequencies:
            cond = f'{intensity}{freq}'
            safe_cond = cond.replace('_', '').replace('-', '')
            fname = deriv / f'sub-{subject}_task-{task}_cond-{safe_cond}_ave.fif'
            if fname.exists():
                try:
                    ev = mne.read_evokeds(fname, verbose=False)[0]
                    evokeds[cond] = ev
                except Exception as e:
                    print(f'  Warning: could not read {fname.name}: {e}')
    return evokeds

## 3. Load component peaks from JSON

Peak latencies and analysis windows are read from `outputs/component_peaks.json`
written by notebook 02, and expanded across all subjects and reference types
defined in `REFERENCES` above.

Also loads per-intensity latencies (`subjects_by_intensity`, notebook 02 Section 3a) into
`peak_latencies_by_intensity` / `analysis_windows_by_intensity` — latencies determined
separately within each intensity level (8 frequencies each), used by Section 6's
per-intensity plots below. `peak_latencies` / `analysis_windows` (24-condition basis)
remain the only basis used everywhere else (Sections 4, 7, 8, 9).


In [ ]:
_json_path = Path('/workspace/outputs/component_peaks.json')
if not _json_path.exists():
    raise FileNotFoundError(
        f'{_json_path} not found.\n'
        'Run notebook 02 (Setup cells 0-4) to generate it.'
    )

with open(_json_path) as _f:
    _comp_data = json.load(_f)

# Warn if any SUBJECTS are missing from the JSON
_missing_subjs = set(SUBJECTS) - set(_comp_data['subjects'])
if _missing_subjs:
    print(f'WARNING: subjects not in JSON: {_missing_subjs}')
    print(f'  Re-run notebook 02 with all subjects to update the JSON.')

# Reconstruct peak_latencies and analysis_windows for all subjects × references
# The same latencies (detected from linkedEars, equal-weighted across all 24
# conditions — see notebook 02 Section 3) are applied to all refs.
peak_latencies  = {}
analysis_windows = {}

for _subj in SUBJECTS:
    if _subj not in _comp_data['subjects']:
        continue
    for ref_label in REFERENCES:
        for comp_name, _vals in _comp_data['subjects'][_subj].items():
            lat = _vals['latency_ms']
            w0  = _vals['window_tmin_ms']
            w1  = _vals['window_tmax_ms']
            lat = float('nan') if lat is None else float(lat)
            w0  = float('nan') if w0  is None else float(w0)
            w1  = float('nan') if w1  is None else float(w1)
            peak_latencies[  (_subj, ref_label, comp_name)] = lat
            analysis_windows[(_subj, ref_label, comp_name)] = (w0, w1)

# Per-intensity latencies (notebook 02, Section 3a) — only latency_ms is saved,
# so the analysis window is recomputed here from COMPONENT_WINDOW_MS, the same
# half-widths used for the 24-condition-basis windows above.
peak_latencies_by_intensity   = {}
analysis_windows_by_intensity = {}

_by_intensity = _comp_data.get('subjects_by_intensity', {})
for _subj in SUBJECTS:
    if _subj not in _by_intensity:
        continue
    for _intensity, _comps in _by_intensity[_subj].items():
        for comp_name, _vals in _comps.items():
            lat = _vals['latency_ms']
            lat = float('nan') if lat is None else float(lat)
            peak_latencies_by_intensity[(_subj, _intensity, comp_name)] = lat
            if np.isnan(lat):
                analysis_windows_by_intensity[(_subj, _intensity, comp_name)] = (np.nan, np.nan)
            else:
                half = COMPONENT_WINDOW_MS[comp_name]
                analysis_windows_by_intensity[(_subj, _intensity, comp_name)] = (lat - half, lat + half)

print(f'Loaded: {_json_path}')
print(f'  Source: ref={_comp_data["latency_ref"]}, '
      f'intensities={_comp_data["latency_intensities"]}, '
      f'frequencies={len(_comp_data["latency_frequencies"])} freqs')
print()
print(f'{"Subject":<10} {"Ref":<12} {"Comp":<5}  {"Latency":>9}  {"Win tmin":>9}  {"Win tmax":>9}')
print('-' * 75)
for (_subj, _ref, comp_name), (_w0, _w1) in analysis_windows.items():
    _lat = peak_latencies[(_subj, _ref, comp_name)]
    _lat_s = f'{_lat:8.1f}' if not np.isnan(_lat) else '     n/a'
    _w0_s  = f'{_w0:9.1f}' if not np.isnan(_w0)  else '     n/a'
    _w1_s  = f'{_w1:9.1f}' if not np.isnan(_w1)  else '     n/a'
    print(f'{_subj:<10} {_ref:<12} {comp_name:<5}  {_lat_s}  {_w0_s}  {_w1_s}')

if peak_latencies_by_intensity:
    print()
    print('Per-intensity latencies (notebook 02, Section 3a):')
    print(f'{"Subject":<10} {"Intensity":<10} {"Comp":<5}  {"Latency":>9}  {"Win tmin":>9}  {"Win tmax":>9}')
    print('-' * 75)
    for (_subj, _intensity, comp_name), (_w0, _w1) in analysis_windows_by_intensity.items():
        _lat = peak_latencies_by_intensity[(_subj, _intensity, comp_name)]
        _lat_s = f'{_lat:8.1f}' if not np.isnan(_lat) else '     n/a'
        _w0_s  = f'{_w0:9.1f}' if not np.isnan(_w0)  else '     n/a'
        _w1_s  = f'{_w1:9.1f}' if not np.isnan(_w1)  else '     n/a'
        print(f'{_subj:<10} {_intensity:<10} {comp_name:<5}  {_lat_s}  {_w0_s}  {_w1_s}')


## 4. Step 2 — Measure amplitudes at fixed latencies across all conditions

In [ ]:
all_rows = []

for ref_label, (pipeline, gfp_exclude, amp_exclude) in REFERENCES.items():
    print(f'\n=== reference: {ref_label} ===')

    for subject in SUBJECTS:
        # Only load 75 dB — latencies were determined from this condition
        evokeds = load_evokeds_for_subject(
            BIDS_ROOT, subject, pipeline, [LATENCY_INTENSITY], FREQUENCIES
        )
        if not evokeds:
            print(f'  sub-{subject}: no evoked files — skipping')
            continue

        print(f'  sub-{subject}: {len(evokeds)} conditions')

        for cond_label, evoked in evokeds.items():
            try:
                db_end    = cond_label.index('dB') + 2
                intensity = cond_label[:db_end]
                frequency = cond_label[db_end:]

                all_picks = get_all_eeg_picks(evoked, gfp_exclude)
                gfp       = compute_gfp(evoked.data[all_picks] * 1e6)

                sig_picks = get_signal_picks(evoked, amp_exclude)
                sig_data  = evoked.data[sig_picks] * 1e6
                times_ms  = evoked.times * 1000

                for comp_name, polarity, tmin_ms, tmax_ms in COMPONENTS:
                    lat_ms = peak_latencies.get((subject, ref_label, comp_name), np.nan)

                    if np.isnan(lat_ms):
                        gfp_amp = ch_name = ch_amp = np.nan
                    else:
                        t_idx   = int(np.argmin(np.abs(times_ms - lat_ms)))
                        gfp_amp = float(gfp[t_idx])
                        col     = sig_data[:, t_idx]
                        ch_idx  = int(np.argmax(np.abs(col)))
                        ch_name = evoked.ch_names[sig_picks[ch_idx]]
                        ch_amp  = float(col[ch_idx])

                    all_rows.append(dict(
                        subject=subject,
                        reference=ref_label,
                        condition=cond_label,
                        intensity=intensity,
                        frequency=frequency,
                        component=comp_name,
                        search_tmin_ms=tmin_ms,
                        search_tmax_ms=tmax_ms,
                        latency_ms=lat_ms,
                        gfp_amplitude_uv=gfp_amp,
                        peak_channel=ch_name,
                        peak_channel_amplitude_uv=ch_amp,
                        nave=evoked.nave,
                    ))
            except Exception as e:
                print(f'  Warning: sub-{subject} {cond_label} failed: {e}')
                continue

df = pd.DataFrame(all_rows)
print(f'\nTotal rows: {len(df)}')
df.head(10)

## 5. Save to TSV

In [ ]:
# Per-subject per-reference files
for ref_label in df['reference'].unique():
    for subject in df['subject'].unique():
        subset = df[(df['reference'] == ref_label) & (df['subject'] == subject)]
        if subset.empty:
            continue
        fname = OUTPUT_DIR / f'sub-{subject}_ref-{ref_label}_peak-latencies.tsv'
        subset.to_csv(fname, sep='\t', index=False)
        print(f'Saved: {fname.relative_to(REPO_ROOT)}  ({len(subset)} rows)')

# Combined file
combined_path = OUTPUT_DIR / 'all_subjects_peak-latencies.tsv'
df.to_csv(combined_path, sep='\t', index=False)
print(f'Saved: {combined_path.relative_to(REPO_ROOT)}  (combined, {len(df)} rows)')

## 6. GFP plot with component markers, per intensity level

Visual sanity check — one figure per intensity level. Each figure shows the
equal-weighted GFP grand average within that intensity (8 frequency conditions,
averaged with equal weight per condition — same procedure as notebook 02 Section 3a),
with that intensity's own peak latencies and analysis windows overlaid (from
`peak_latencies_by_intensity` / `analysis_windows_by_intensity`, Section 3 above) —
not the 24-condition-basis latencies used elsewhere in this notebook.
Edit `PLOT_SUBJECT` and `PLOT_REF` to inspect different subjects/references.


In [ ]:
PLOT_SUBJECT = 'Cj399'
PLOT_REF     = 'linkedEars'

pipeline, gfp_exclude, amp_exclude = REFERENCES[PLOT_REF]

comp_colors = {'P1': '#2196F3', 'N1': '#F44336', 'P2': '#4CAF50',
               'N2': '#FF9800', 'LN': '#9C27B0'}

SAVE_DIR = '/workspace/figures'
import ipywidgets as _w
from IPython.display import display as _display

for intensity in INTENSITIES:
    evokeds_int = load_evokeds_for_subject(
        BIDS_ROOT, PLOT_SUBJECT, pipeline, [intensity], FREQUENCIES
    )
    if not evokeds_int:
        print(f'No evokeds for sub-{PLOT_SUBJECT}, ref-{PLOT_REF}, {intensity}')
        continue

    all_picks  = get_all_eeg_picks(next(iter(evokeds_int.values())), gfp_exclude)
    mean_gfp   = compute_condition_gfp_grand_average(evokeds_int, all_picks)
    ref_evoked = next(iter(evokeds_int.values()))
    times_ms   = ref_evoked.times * 1000
    n_cond     = len(evokeds_int)
    n_expected = len(FREQUENCIES)

    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.plot(times_ms, mean_gfp, color='k', lw=1.5,
            label=f'GFP ({n_cond}/{n_expected} conditions, equal-weighted)')
    ax.axvline(0, color='gray', lw=0.8, linestyle='--')
    ax.set_xlim(-100, 650)
    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('GFP (µV)')
    ax.set_title(f'sub-{PLOT_SUBJECT}  ref-{PLOT_REF}  {intensity}  '
                 f'grand-average GFP ({n_cond}/{n_expected} conditions)  '
                 f'(all EEG ch, ddof=0)')

    for comp_name, polarity, tmin_ms, tmax_ms in COMPONENTS:
        col = comp_colors.get(comp_name, 'gray')
        lat = peak_latencies_by_intensity.get((PLOT_SUBJECT, intensity, comp_name), np.nan)
        if np.isnan(lat):
            continue

        w0, w1 = analysis_windows_by_intensity.get(
            (PLOT_SUBJECT, intensity, comp_name), (np.nan, np.nan)
        )
        if not np.isnan(w0):
            ax.axvspan(w0, w1, alpha=0.15, color=col, zorder=1)

        amp = float(mean_gfp[np.argmin(np.abs(times_ms - lat))])
        ax.axvline(lat, color=col, lw=1.2, linestyle=':', zorder=2)
        ax.text(lat + 2, amp + 0.05, comp_name, color=col, fontsize=8,
                va='bottom', zorder=3)

    fig.tight_layout()
    plt.show()

    _default_name = f'sub-{PLOT_SUBJECT}_ref-{PLOT_REF}_{intensity}_GFP'
    _name = _w.Text(value=_default_name,
                    description='Filename:', layout=_w.Layout(width='360px'))
    _fmt  = _w.Dropdown(options=['pdf', 'svg', 'png'], value='pdf',
                        description='Format:', layout=_w.Layout(width='160px'))
    _btn  = _w.Button(description='Save', button_style='primary',
                      layout=_w.Layout(width='80px'))
    _out  = _w.Output()

    def _save(_, f=fig, n=_name, fmt=_fmt, o=_out):
        with o:
            o.clear_output()
            path = f'{SAVE_DIR}/{n.value}.{fmt.value}'
            f.savefig(path, format=fmt.value, bbox_inches='tight')
            print(f'Saved: {path}')

    _btn.on_click(_save)
    _display(_w.HBox([_name, _fmt, _btn]), _out)


## 6a. GFP plot with component markers, all conditions pooled

Grand-average GFP across all 24 conditions (3 intensities × 8 frequencies,
equal-weighted — same procedure as notebook 02 Section 3), overlaid with the
global 24-condition-basis peak latencies and analysis windows (`peak_latencies` /
`analysis_windows`, Section 3 above) — the same latencies used everywhere else in
this notebook (Sections 4, 7, 8, 9). Unlike Section 6's per-intensity plots, this
is the single grand average pooling all intensities and frequencies together.
Edit `PLOT6A_SUBJECT` and `PLOT6A_REF` to inspect different subjects/references.

In [ ]:
PLOT6A_SUBJECT = 'Cj459'
PLOT6A_REF     = 'linkedEars'

pipeline, gfp_exclude, amp_exclude = REFERENCES[PLOT6A_REF]

comp_colors = {'P1': '#2196F3', 'N1': '#F44336', 'P2': '#4CAF50',
               'N2': '#FF9800', 'LN': '#9C27B0'}

SAVE_DIR = '/workspace/figures'
import ipywidgets as _w
from IPython.display import display as _display

evokeds_all = load_evokeds_for_subject(
    BIDS_ROOT, PLOT6A_SUBJECT, pipeline, INTENSITIES, FREQUENCIES
)
if not evokeds_all:
    print(f'No evokeds for sub-{PLOT6A_SUBJECT}, ref-{PLOT6A_REF}')
else:
    all_picks  = get_all_eeg_picks(next(iter(evokeds_all.values())), gfp_exclude)
    mean_gfp   = compute_condition_gfp_grand_average(evokeds_all, all_picks)
    ref_evoked = next(iter(evokeds_all.values()))
    times_ms   = ref_evoked.times * 1000
    n_cond     = len(evokeds_all)
    n_expected = len(INTENSITIES) * len(FREQUENCIES)

    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.plot(times_ms, mean_gfp, color='k', lw=1.5,
            label=f'GFP ({n_cond}/{n_expected} conditions, equal-weighted)')
    ax.axvline(0, color='gray', lw=0.8, linestyle='--')
    ax.set_xlim(-100, 650)
    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('GFP (µV)')
    ax.set_title(f'sub-{PLOT6A_SUBJECT}  ref-{PLOT6A_REF}  '
                 f'grand-average GFP ({n_cond}/{n_expected} conditions, all intensities)  '
                 f'(all EEG ch, ddof=0)')

    for comp_name, polarity, tmin_ms, tmax_ms in COMPONENTS:
        col = comp_colors.get(comp_name, 'gray')
        lat = peak_latencies.get((PLOT6A_SUBJECT, PLOT6A_REF, comp_name), np.nan)
        if np.isnan(lat):
            continue

        w0, w1 = analysis_windows.get((PLOT6A_SUBJECT, PLOT6A_REF, comp_name), (np.nan, np.nan))
        if not np.isnan(w0):
            ax.axvspan(w0, w1, alpha=0.15, color=col, zorder=1)

        amp = float(mean_gfp[np.argmin(np.abs(times_ms - lat))])
        ax.axvline(lat, color=col, lw=1.2, linestyle=':', zorder=2)
        ax.text(lat + 2, amp + 0.05, comp_name, color=col, fontsize=8,
                ha='left', va='bottom')

    ax.legend(loc='upper right', fontsize=8)
    fig.tight_layout()
    plt.show()

    _default_name = f'sub-{PLOT6A_SUBJECT}_ref-{PLOT6A_REF}_GFP-allconditions'
    _name = _w.Text(value=_default_name,
                    description='Filename:', layout=_w.Layout(width='360px'))
    _fmt  = _w.Dropdown(options=['pdf', 'svg', 'png'], value='pdf',
                        description='Format:', layout=_w.Layout(width='160px'))
    _btn  = _w.Button(description='Save', button_style='primary',
                      layout=_w.Layout(width='80px'))
    _out  = _w.Output()

    def _save(_, f=fig, n=_name, fmt=_fmt, o=_out):
        with o:
            o.clear_output()
            path = f'{SAVE_DIR}/{n.value}.{fmt.value}'
            f.savefig(path, format=fmt.value, bbox_inches='tight')
            print(f'Saved: {path}')

    _btn.on_click(_save)
    _display(_w.HBox([_name, _fmt, _btn]), _out)


## 6b. GFP overlay by intensity level

One line per intensity (GFP averaged across all 8 frequency conditions).  
Component windows and peak latencies are taken from Section 3.  
Edit `PLOT7B_SUBJECT` and `PLOT7B_REF` to switch subject or reference.

In [ ]:
# ── Section 7b settings ───────────────────────────────────────────────────────
PLOT7B_SUBJECT = 'Cj399'
PLOT7B_REF     = 'linkedEars'   # must be a key in REFERENCES
# ─────────────────────────────────────────────────────────────────────────────
SAVE_DIR = '/workspace/figures'

_int_colors_7b  = {'45dB': '#2196F3', '60dB': '#FF9800', '75dB': '#F44336'}
_comp_colors_7b = {'P1': '#2196F3', 'N1': '#F44336', 'P2': '#4CAF50',
                   'N2': '#FF9800', 'LN': '#9C27B0'}

pipeline, gfp_exclude, amp_exclude = REFERENCES[PLOT7B_REF]

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.axvline(0, color='gray', lw=0.8, linestyle='--')
ax.set_xlim(-100, 650)

ref_ev_7b = None
for intensity in INTENSITIES:
    evokeds = load_evokeds_for_subject(
        BIDS_ROOT, PLOT7B_SUBJECT, pipeline, [intensity], FREQUENCIES
    )
    if not evokeds:
        print(f'No evokeds for sub-{PLOT7B_SUBJECT} {intensity} — skipping')
        continue

    # Average GFP across all 8 frequency conditions (mirrors Section 3 approach)
    gfp_stack = []
    for ev in evokeds.values():
        picks = get_all_eeg_picks(ev, gfp_exclude)
        gfp_stack.append(compute_gfp(ev.data[picks] * 1e6))
        ref_ev_7b = ev
    mean_gfp_7b = np.mean(gfp_stack, axis=0)
    ax.plot(ref_ev_7b.times * 1000, mean_gfp_7b,
            color=_int_colors_7b[intensity], lw=1.8, label=intensity)

if ref_ev_7b is not None:
    times_ms_7b = ref_ev_7b.times * 1000
    for comp_name, polarity, tmin_ms, tmax_ms in COMPONENTS:
        col = _comp_colors_7b.get(comp_name, 'gray')
        lat = peak_latencies.get((PLOT7B_SUBJECT, PLOT7B_REF, comp_name), np.nan)
        if np.isnan(lat):
            continue
        w0, w1 = analysis_windows.get(
            (PLOT7B_SUBJECT, PLOT7B_REF, comp_name), (np.nan, np.nan)
        )
        if not np.isnan(w0):
            ax.axvspan(w0, w1, alpha=0.12, color=col, zorder=1)
        ax.axvline(lat, color=col, lw=1.2, linestyle=':', zorder=2)
        ax.text(lat + 2, 0.97, comp_name, color=col, fontsize=8, va='top',
                transform=ax.get_xaxis_transform(), zorder=3)

ax.set_xlabel('Time (ms)')
ax.set_ylabel('GFP (µV)')
ax.set_title(
    f'sub-{PLOT7B_SUBJECT}  ref-{PLOT7B_REF}  — GFP by intensity (avg over {len(FREQUENCIES)} frequencies)'
)
ax.legend(title='Intensity', fontsize=9, title_fontsize=9)
fig.tight_layout()
plt.show()


_default_name = f'sub-{PLOT7B_SUBJECT}_ref-{PLOT7B_REF}_GFP-by-intensity'
import ipywidgets as _w
from IPython.display import display as _display
_name = _w.Text(value=_default_name,
                description='Filename:', layout=_w.Layout(width='360px'))
_fmt  = _w.Dropdown(options=['pdf', 'svg', 'png'], value='pdf',
                    description='Format:', layout=_w.Layout(width='160px'))
_btn  = _w.Button(description='Save', button_style='primary',
                  layout=_w.Layout(width='80px'))
_out  = _w.Output()
_fig_ref = fig

def _save(_):
    with _out:
        _out.clear_output()
        path = f'{SAVE_DIR}/{_name.value}.{_fmt.value}'
        _fig_ref.savefig(path, format=_fmt.value, bbox_inches='tight')
        print(f'Saved: {path}')

_btn.on_click(_save)
_display(_w.HBox([_name, _fmt, _btn]), _out)


## 6c. GFP overlay by frequency — one figure per intensity

One figure per intensity level; each figure overlays GFP waveforms for all 8 frequency conditions.
Component windows and peak latencies are taken from Section 3.
Edit `PLOT6C_SUBJECT` and `PLOT6C_REF` to switch subject or reference.

In [ ]:
# ── Section 6c settings ───────────────────────────────────────────────────────
PLOT6C_SUBJECT = 'Cj399'
PLOT6C_REF     = 'linkedEars'   # must be a key in REFERENCES
# ─────────────────────────────────────────────────────────────────────────────
SAVE_DIR = '/workspace/figures'

_freq_colors_6c = {
    f: c for f, c in zip(FREQUENCIES, [
        '#e6194b','#3cb44b','#4363d8','#f58231',
        '#911eb4','#42d4f4','#f032e6','#bfef45',
    ])
}
_comp_colors_6c = {'P1': '#2196F3', 'N1': '#F44336', 'P2': '#4CAF50',
                   'N2': '#FF9800', 'LN': '#9C27B0'}

pipeline6c, gfp_exclude_6c, amp_exclude_6c = REFERENCES[PLOT6C_REF]

# Compute y-axis limits from the 75 dB condition so all figures share the same scale
_ev75_all = load_evokeds_for_subject(
    BIDS_ROOT, PLOT6C_SUBJECT, pipeline6c, ['75dB'], FREQUENCIES
)
if _ev75_all:
    _gfp75_vals = []
    for _ev in _ev75_all.values():
        _picks75 = get_all_eeg_picks(_ev, gfp_exclude_6c)
        _gfp75_vals.append(compute_gfp(_ev.data[_picks75] * 1e6))
    _gfp75_all = np.concatenate(_gfp75_vals)
    _pad75 = (_gfp75_all.max() - _gfp75_all.min()) * 0.05
    _ylim6c = (_gfp75_all.min() - _pad75, _gfp75_all.max() + _pad75)
else:
    _ylim6c = None

import ipywidgets as _w
from IPython.display import display as _display

for intensity in INTENSITIES:
    evokeds = load_evokeds_for_subject(
        BIDS_ROOT, PLOT6C_SUBJECT, pipeline6c, [intensity], FREQUENCIES
    )
    if not evokeds:
        print(f'No evokeds for sub-{PLOT6C_SUBJECT} {intensity} — skipping')
        continue

    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.axvline(0, color='gray', lw=0.8, linestyle='--')
    ax.set_xlim(-100, 650)
    if _ylim6c is not None:
        ax.set_ylim(_ylim6c)

    ref_ev_6c = None
    for freq in FREQUENCIES:
        cond_label = f'{intensity}{freq}'
        ev = evokeds.get(cond_label)
        if ev is None:
            continue
        picks = get_all_eeg_picks(ev, gfp_exclude_6c)
        gfp   = compute_gfp(ev.data[picks] * 1e6)
        hz    = int(freq.replace('Hz', ''))
        lbl   = f'{hz // 1000}k Hz' if hz >= 1000 else f'{hz} Hz'
        ax.plot(ev.times * 1000, gfp,
                color=_freq_colors_6c[freq], lw=1.4, label=lbl)
        ref_ev_6c = ev

    if ref_ev_6c is not None:
        times_ms_6c = ref_ev_6c.times * 1000
        for comp_name, polarity, tmin_ms, tmax_ms in COMPONENTS:
            col = _comp_colors_6c.get(comp_name, 'gray')
            lat = peak_latencies.get((PLOT6C_SUBJECT, PLOT6C_REF, comp_name), np.nan)
            if np.isnan(lat):
                continue
            w0, w1 = analysis_windows.get(
                (PLOT6C_SUBJECT, PLOT6C_REF, comp_name), (np.nan, np.nan)
            )
            if not np.isnan(w0):
                ax.axvspan(w0, w1, alpha=0.12, color=col, zorder=1)
            ax.axvline(lat, color=col, lw=1.2, linestyle=':', zorder=2)
            ax.text(lat + 2, 0.97, comp_name, color=col, fontsize=8,
                    va='top', transform=ax.get_xaxis_transform(), zorder=3)

    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('GFP (µV)')
    ax.set_title(
        f'sub-{PLOT6C_SUBJECT}  ref-{PLOT6C_REF}  {intensity}'
        f' — GFP by frequency'
    )
    ax.legend(title='Frequency', fontsize=8, title_fontsize=8,
              loc='upper right', ncol=2)
    fig.tight_layout()
    plt.show()

    _default_name = f'sub-{PLOT6C_SUBJECT}_ref-{PLOT6C_REF}_GFP-by-freq_{intensity}'
    _name = _w.Text(value=_default_name,
                    description='Filename:', layout=_w.Layout(width='360px'))
    _fmt  = _w.Dropdown(options=['pdf', 'svg', 'png'], value='pdf',
                        description='Format:', layout=_w.Layout(width='160px'))
    _btn  = _w.Button(description='Save', button_style='primary',
                      layout=_w.Layout(width='80px'))
    _out  = _w.Output()

    def _save(_, _fig=fig, _out=_out, _name=_name, _fmt=_fmt):
        with _out:
            _out.clear_output()
            path = f'{SAVE_DIR}/{_name.value}.{_fmt.value}'
            _fig.savefig(path, format=_fmt.value, bbox_inches='tight')
            print(f'Saved: {path}')

    _btn.on_click(_save)
    _display(_w.HBox([_name, _fmt, _btn]), _out)


---

## 7. Linear mixed model (Intensity × Frequency, random intercept for Run): GFP

Runs on **linked-ears data only** (CAR/CMR are not analyzed here — component peak
latencies were derived from linkedEars GFP alone, notebook 02 Section 3, and this
section tests effects at those latencies). For each **subject × component**:

1. Load clean epoch files (one per intensity level; each carries a `run` metadata column
   identifying which of the 4 recording blocks — i.e. cycle — a trial came from).
2. For each trial, compute mean GFP over the component's analysis window.
3. Fit a linear mixed model:

   `GFP ~ C(intensity) * C(frequency) + (1 | run_id)`

   - `intensity`: 3-level categorical (45/60/75 dB)
   - `frequency`: 8-level categorical
   - `run_id`: random-intercept grouping factor with 12 levels — one per
     (intensity, cycle) recording block, e.g. `75dB_run1`, `75dB_run2`, ... `45dB_run4`
     (Cj399 cycles 75→60→45 dB, Cj459 cycles 45→60→75 dB, 4 times each)

**Type III omnibus test per factor**: in addition to the individual dummy-coded
coefficients above, a joint Wald chi-square test (`fit.wald_test_terms(scalar=True)`)
is run for each of `C(intensity)`, `C(frequency)`, and `C(intensity):C(frequency)` as a
whole — each term tested conditional on every other term already in the model, matching
a Type III ANOVA's convention (valid here because `C()` uses dummy coding). This is an
asymptotic χ² test. `C(intensity)` only varies *between* runs, its joint test draws
on the same random-intercept-competing covariance that produces NaN/blown-up per-coefficient
SEs when `Group Var≈0` (see above) — so whenever that boundary flag appears, the
`C(intensity)` omnibus row is marked `unreliable_at_boundary=True` rather than trusted;
Section 8's run-level OLS is the reliable source for the intensity effect in that case.
`C(frequency)` and the interaction term are unaffected (orthogonal to the random-effect
structure) and can be read normally regardless of the boundary flag.

Output: `outputs/stats/sub-<ID>_ref-<ref>_lmm_GFP.tsv` + combined `all_subjects_lmm_GFP.tsv`
(per-coefficient results), and `sub-<ID>_ref-<ref>_lmm_omnibus_GFP.tsv` + combined
`all_subjects_lmm_omnibus_GFP.tsv` (Type III omnibus results)

Columns (per-coefficient): `reference`, `subject`, `component`, `term`, `coef`, `std_err`, `z`, `p`, `ci_lower`, `ci_upper`

Columns (omnibus): `reference`, `subject`, `component`, `term`, `chi2`, `df`, `p`, `unreliable_at_boundary`

In [ ]:
def load_epochs_for_subject(bids_root, subject, pipeline, intensities):
    """Load clean epoch files; return {intensity: mne.Epochs}. Missing files skipped."""
    deriv = Path(bids_root) / 'derivatives' / pipeline / f'sub-{subject}' / 'eeg'
    epochs_by_intensity = {}
    for intensity in intensities:
        fif = deriv / f'sub-{subject}_task-Passive{intensity}_desc-clean_epo.fif'
        if fif.exists():
            epo = mne.read_epochs(fif, preload=True, verbose=False)
            epochs_by_intensity[intensity] = epo
        else:
            print(f'  Missing epoch file: {fif.name}')
    return epochs_by_intensity


def mean_gfp_over_window(epochs, all_picks, tmin_ms, tmax_ms):
    """
    Return mean GFP amplitude (µV) per trial over [tmin_ms, tmax_ms].
    epochs.get_data() shape: (n_trials, n_ch, n_times)
    """
    times_ms = epochs.times * 1000
    mask = (times_ms >= tmin_ms) & (times_ms <= tmax_ms)
    data = epochs.get_data(picks=all_picks) * 1e6   # (n_trials, n_ch, n_times)
    # GFP per trial per time: std across channels (ddof=0)
    gfp = data.std(axis=1, ddof=0)                  # (n_trials, n_times)
    return gfp[:, mask].mean(axis=1)                # (n_trials,)


def mean_erp_over_window(epochs, ch_picks, tmin_ms, tmax_ms):
    """
    Return mean signed ERP amplitude (µV) per trial over [tmin_ms, tmax_ms],
    for one or more single channels (not a GFP across channels).
    epochs.get_data() shape: (n_trials, n_ch_picks, n_times)
    """
    times_ms = epochs.times * 1000
    mask = (times_ms >= tmin_ms) & (times_ms <= tmax_ms)
    data = epochs.get_data(picks=ch_picks) * 1e6    # (n_trials, n_ch_picks, n_times)
    return data[:, :, mask].mean(axis=2)            # (n_trials, n_ch_picks)

In [ ]:
lmm_rows = []
omnibus_rows = []

# Section 7 (and 7c/7d) run on linked-ears data only — CAR/CMR are not analyzed
# here, since component peak latencies were derived from linkedEars GFP alone
# (notebook 02, Section 3) and this section tests effects at those latencies.
SEC7_REFERENCES = {'linkedEars': REFERENCES['linkedEars']}

for ref_label, (pipeline, gfp_exclude, amp_exclude) in SEC7_REFERENCES.items():
    print(f'\n=== reference: {ref_label} ===')

    for subject in SUBJECTS:
        print(f'\n  sub-{subject}')
        epochs_by_intensity = load_epochs_for_subject(
            BIDS_ROOT, subject, pipeline, INTENSITIES
        )
        if len(epochs_by_intensity) < 2:
            print(f'  sub-{subject}: fewer than 2 intensity levels loaded — skipping LMM')
            continue

        ref_epo   = next(iter(epochs_by_intensity.values()))
        all_picks = get_all_eeg_picks(ref_epo)

        for comp_name, polarity, search_tmin, search_tmax in COMPONENTS:
            w0, w1 = analysis_windows.get((subject, ref_label, comp_name), (np.nan, np.nan))
            if np.isnan(w0):
                print(f'    {comp_name}: no analysis window — skipping')
                continue

            # Build long-form DataFrame: one row per trial, with run_id recovered
            # from the 'run' column added to epochs.metadata in notebook 01 (one
            # of the 4 recording blocks/cycles per intensity). run_id is the
            # (intensity, cycle) cross — 12 unique recording blocks.
            rows = []
            for intensity, epo in epochs_by_intensity.items():
                for freq in FREQUENCIES:
                    cond_label = f'{intensity}{freq}'
                    if cond_label not in epo.event_id:
                        continue
                    cond_epo = epo[cond_label]
                    if len(cond_epo) == 0:
                        continue
                    amps  = mean_gfp_over_window(cond_epo, all_picks, w0, w1)
                    cycle = cond_epo.metadata['run'].to_numpy()
                    for amp, cyc in zip(amps, cycle):
                        rows.append(dict(
                            intensity=intensity, frequency=freq,
                            run_id=f'{intensity}_run{cyc}',
                            amplitude=amp,
                        ))

            if not rows:
                print(f'    {comp_name}: no data — skipping')
                continue

            long_df = pd.DataFrame(rows)

            try:
                # No separate fixed effect for cycle: run_id already is the
                # (intensity, cycle) cross, so a C(cycle) term would compete with
                # the (1 | run_id) random intercept for the same between-run
                # variance (see Section 7 markdown for why that caused a
                # singular random-effects covariance).
                model = smf.mixedlm(
                    'amplitude ~ C(intensity) * C(frequency)',
                    data=long_df,
                    groups=long_df['run_id'],
                )
                # lbfgs is used directly (skipping the default powell start) since
                # powell reliably fails to converge on this design (many fixed-effect
                # levels relative to per-run trial counts) and only adds noisy
                # ConvergenceWarnings before statsmodels retries with lbfgs anyway.
                # Boundary-solution warnings ("singular", "boundary of parameter
                # space") are expected whenever Group Var lands at ~0 — they are
                # summarised as a single "[boundary]" flag below instead of being
                # left to print their raw multi-line text.
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter('always')
                    fit = model.fit(reml=True, method='lbfgs')
                boundary = any(
                    'singular' in str(w.message) or 'boundary' in str(w.message)
                    for w in caught
                )

                for term, coef in fit.params.items():
                    if term == 'Group Var':
                        continue
                    lmm_rows.append(dict(
                        reference=ref_label,
                        subject=subject,
                        component=comp_name,
                        term=term,
                        coef=round(float(coef), 6),
                        std_err=round(float(fit.bse[term]), 6),
                        z=round(float(fit.tvalues[term]), 4),
                        p=round(float(fit.pvalues[term]), 6),
                        ci_lower=round(float(fit.conf_int().loc[term, 0]), 6),
                        ci_upper=round(float(fit.conf_int().loc[term, 1]), 6),
                    ))

                # Type III-style omnibus test per factor: a joint Wald chi-square
                # test of all dummy-coded levels of each term at once (each term
                # tested conditional on every other term already in the model,
                # since C() uses dummy coding — matching a Type III ANOVA's
                # "each term last" convention). C(intensity) only varies
                # *between* runs, so at the Group Var≈0 boundary its joint test
                # draws on the same contaminated covariance that produces the
                # NaN/blown-up per-coefficient SEs above — see Section 8's
                # markdown for why intensity effects are tested via run-level
                # OLS there instead. That row is flagged, not dropped, so it is
                # visible but clearly marked as unreliable at the boundary.
                wald = fit.wald_test_terms(scalar=True)
                for term_name in ['C(intensity)', 'C(frequency)', 'C(intensity):C(frequency)']:
                    row = wald.table.loc[term_name]
                    unreliable = boundary and term_name == 'C(intensity)'
                    omnibus_rows.append(dict(
                        reference=ref_label,
                        subject=subject,
                        component=comp_name,
                        term=term_name,
                        chi2=round(float(row['statistic']), 4),
                        df=int(row['df_constraint']),
                        p=round(float(row['pvalue']), 6),
                        unreliable_at_boundary=unreliable,
                    ))

                # Group Var ~0 means the between-run random-intercept variance is
                # negligible once intensity/frequency are accounted for — a valid
                # result, not a fitting failure (see converged flag). The
                # "[boundary]" flag marks exactly this case, in place of the
                # underlying multi-line singular-covariance warnings.
                group_var = float(fit.params['Group Var'])
                n_sig = int((fit.pvalues.drop('Group Var', errors='ignore') < 0.05).sum())
                flag = '  [boundary: Group Var≈0 — C(intensity) omnibus unreliable]' if boundary else ''
                print(f'    {comp_name}: converged={fit.converged}  '
                      f'Group Var={group_var:.6g}  '
                      f'{n_sig}/{len(fit.params) - 1} terms p<.05{flag}')
            except Exception as e:
                print(f'    {comp_name}: LMM failed — {e}')

lmm_df = pd.DataFrame(lmm_rows)
print(f'\nTotal LMM result rows: {len(lmm_df)}')

omnibus_df = pd.DataFrame(omnibus_rows)
print(f'Total omnibus (Type III Wald) result rows: {len(omnibus_df)}')
lmm_df.head(15)

### 7b. Save LMM results to TSV

In [ ]:
# Per-subject per-reference files
for ref_label in lmm_df['reference'].unique():
    for subject in lmm_df['subject'].unique():
        subset = lmm_df[(lmm_df['reference'] == ref_label) & (lmm_df['subject'] == subject)]
        if subset.empty:
            continue
        fname = OUTPUT_DIR / f'sub-{subject}_ref-{ref_label}_lmm_GFP.tsv'
        subset.to_csv(fname, sep='\t', index=False)
        print(f'Saved: {fname.relative_to(REPO_ROOT)}  ({len(subset)} rows)')

# Combined file
combined_path = OUTPUT_DIR / 'all_subjects_lmm_GFP.tsv'
lmm_df.to_csv(combined_path, sep='\t', index=False)
print(f'Saved: {combined_path.relative_to(REPO_ROOT)}  (combined, {len(lmm_df)} rows)')

# Type III omnibus (Wald joint) test results — per-subject per-reference files
for ref_label in omnibus_df['reference'].unique():
    for subject in omnibus_df['subject'].unique():
        subset = omnibus_df[(omnibus_df['reference'] == ref_label) & (omnibus_df['subject'] == subject)]
        if subset.empty:
            continue
        fname = OUTPUT_DIR / f'sub-{subject}_ref-{ref_label}_lmm_omnibus_GFP.tsv'
        subset.to_csv(fname, sep='\t', index=False)
        print(f'Saved: {fname.relative_to(REPO_ROOT)}  ({len(subset)} rows)')

# Combined file
omnibus_combined_path = OUTPUT_DIR / 'all_subjects_lmm_omnibus_GFP.tsv'
omnibus_df.to_csv(omnibus_combined_path, sep='\t', index=False)
print(f'Saved: {omnibus_combined_path.relative_to(REPO_ROOT)}  (combined, {len(omnibus_df)} rows)')


---

## 7c. Negative control: same LMM on the pre-stimulus baseline window

Same model as Section 7 — `GFP ~ C(intensity) * C(frequency) + (1 | run_id)` — applied to
a single window that is not expected to carry a stimulus-evoked response: the pre-stimulus
baseline, **-100 to 0 ms** (the same window each epoch's baseline correction was computed
over — see notebook 01). If the intensity/frequency/interaction effects seen in Section 7
for P1-LN reflect a genuine stimulus-evoked response rather than an artifact of the
analysis pipeline (e.g. spurious run/cycle structure, multiple-comparison inflation), the
same test on baseline data should come back non-significant. A significant effect here
would flag a problem with the pipeline rather than the auditory response itself.

Output: `outputs/stats/sub-<ID>_ref-<ref>_lmm_baseline_GFP.tsv` +
combined `all_subjects_lmm_baseline_GFP.tsv` (per-coefficient), and
`sub-<ID>_ref-<ref>_lmm_omnibus_baseline_GFP.tsv` +
combined `all_subjects_lmm_omnibus_baseline_GFP.tsv` (Type III omnibus)

Same columns as Section 7/7b.


In [ ]:
BASELINE_WINDOW = ('Baseline', -100.0, 0.0)   # (label, tmin_ms, tmax_ms)

lmm_baseline_rows = []
omnibus_baseline_rows = []

comp_name, w0, w1 = BASELINE_WINDOW

for ref_label, (pipeline, gfp_exclude, amp_exclude) in SEC7_REFERENCES.items():
    print(f'\n=== reference: {ref_label} ===')

    for subject in SUBJECTS:
        print(f'\n  sub-{subject}')
        epochs_by_intensity = load_epochs_for_subject(
            BIDS_ROOT, subject, pipeline, INTENSITIES
        )
        if len(epochs_by_intensity) < 2:
            print(f'  sub-{subject}: fewer than 2 intensity levels loaded — skipping LMM')
            continue

        ref_epo   = next(iter(epochs_by_intensity.values()))
        all_picks = get_all_eeg_picks(ref_epo)

        # Same long-form construction as Section 7, but with a single fixed
        # window (the pre-stimulus baseline) instead of looping over COMPONENTS.
        rows = []
        for intensity, epo in epochs_by_intensity.items():
            for freq in FREQUENCIES:
                cond_label = f'{intensity}{freq}'
                if cond_label not in epo.event_id:
                    continue
                cond_epo = epo[cond_label]
                if len(cond_epo) == 0:
                    continue
                amps  = mean_gfp_over_window(cond_epo, all_picks, w0, w1)
                cycle = cond_epo.metadata['run'].to_numpy()
                for amp, cyc in zip(amps, cycle):
                    rows.append(dict(
                        intensity=intensity, frequency=freq,
                        run_id=f'{intensity}_run{cyc}',
                        amplitude=amp,
                    ))

        if not rows:
            print(f'    {comp_name}: no data — skipping')
            continue

        long_df = pd.DataFrame(rows)

        try:
            model = smf.mixedlm(
                'amplitude ~ C(intensity) * C(frequency)',
                data=long_df,
                groups=long_df['run_id'],
            )
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter('always')
                fit = model.fit(reml=True, method='lbfgs')
            boundary = any(
                'singular' in str(w.message) or 'boundary' in str(w.message)
                for w in caught
            )

            for term, coef in fit.params.items():
                if term == 'Group Var':
                    continue
                lmm_baseline_rows.append(dict(
                    reference=ref_label,
                    subject=subject,
                    component=comp_name,
                    term=term,
                    coef=round(float(coef), 6),
                    std_err=round(float(fit.bse[term]), 6),
                    z=round(float(fit.tvalues[term]), 4),
                    p=round(float(fit.pvalues[term]), 6),
                    ci_lower=round(float(fit.conf_int().loc[term, 0]), 6),
                    ci_upper=round(float(fit.conf_int().loc[term, 1]), 6),
                ))

            wald = fit.wald_test_terms(scalar=True)
            for term_name in ['C(intensity)', 'C(frequency)', 'C(intensity):C(frequency)']:
                row = wald.table.loc[term_name]
                unreliable = boundary and term_name == 'C(intensity)'
                omnibus_baseline_rows.append(dict(
                    reference=ref_label,
                    subject=subject,
                    component=comp_name,
                    term=term_name,
                    chi2=round(float(row['statistic']), 4),
                    df=int(row['df_constraint']),
                    p=round(float(row['pvalue']), 6),
                    unreliable_at_boundary=unreliable,
                ))

            group_var = float(fit.params['Group Var'])
            n_sig = int((fit.pvalues.drop('Group Var', errors='ignore') < 0.05).sum())
            flag = '  [boundary: Group Var≈0 — C(intensity) omnibus unreliable]' if boundary else ''
            print(f'    {comp_name}: converged={fit.converged}  '
                  f'Group Var={group_var:.6g}  '
                  f'{n_sig}/{len(fit.params) - 1} terms p<.05{flag}')
        except Exception as e:
            print(f'    {comp_name}: LMM failed — {e}')

lmm_baseline_df = pd.DataFrame(lmm_baseline_rows)
print(f'\nTotal baseline LMM result rows: {len(lmm_baseline_df)}')

omnibus_baseline_df = pd.DataFrame(omnibus_baseline_rows)
print(f'Total baseline omnibus (Type III Wald) result rows: {len(omnibus_baseline_df)}')

# Per-subject per-reference + combined files (same pattern as Section 7b)
for ref_label in lmm_baseline_df['reference'].unique():
    for subject in lmm_baseline_df['subject'].unique():
        subset = lmm_baseline_df[(lmm_baseline_df['reference'] == ref_label) & (lmm_baseline_df['subject'] == subject)]
        if subset.empty:
            continue
        fname = OUTPUT_DIR / f'sub-{subject}_ref-{ref_label}_lmm_baseline_GFP.tsv'
        subset.to_csv(fname, sep='\t', index=False)
        print(f'Saved: {fname.relative_to(REPO_ROOT)}  ({len(subset)} rows)')

combined_path = OUTPUT_DIR / 'all_subjects_lmm_baseline_GFP.tsv'
lmm_baseline_df.to_csv(combined_path, sep='\t', index=False)
print(f'Saved: {combined_path.relative_to(REPO_ROOT)}  (combined, {len(lmm_baseline_df)} rows)')

for ref_label in omnibus_baseline_df['reference'].unique():
    for subject in omnibus_baseline_df['subject'].unique():
        subset = omnibus_baseline_df[(omnibus_baseline_df['reference'] == ref_label) & (omnibus_baseline_df['subject'] == subject)]
        if subset.empty:
            continue
        fname = OUTPUT_DIR / f'sub-{subject}_ref-{ref_label}_lmm_omnibus_baseline_GFP.tsv'
        subset.to_csv(fname, sep='\t', index=False)
        print(f'Saved: {fname.relative_to(REPO_ROOT)}  ({len(subset)} rows)')

omnibus_combined_path = OUTPUT_DIR / 'all_subjects_lmm_omnibus_baseline_GFP.tsv'
omnibus_baseline_df.to_csv(omnibus_combined_path, sep='\t', index=False)
print(f'Saved: {omnibus_combined_path.relative_to(REPO_ROOT)}  (combined, {len(omnibus_baseline_df)} rows)')

lmm_baseline_df.head(15)


---

## 7d. Multiplicative model: log(GFP / baseline GFP), prestimulus baseline as reference

Section 7's model is additive: `GFP ~ C(intensity) * C(frequency) + (1 | run_id)`, so
coefficients are differences in µV. This section instead asks the same question on a
*multiplicative* scale, using each trial's own pre-stimulus baseline (-100 to 0 ms, the
same window as Section 7c) as its reference zero-point:

1. For each trial, compute `ratio = amplitude / baseline_amplitude`, where `amplitude` is
   the component-window GFP (as in Section 7) and `baseline_amplitude` is that same
   trial's baseline-window GFP.
2. Fit the same fixed/random-effect structure on `log(ratio)`:

   `log(ratio) ~ C(intensity) * C(frequency) + (1 | run_id)`

Because baseline GFP is a population std (ddof=0 across channels) it cannot be exactly
zero unless every channel reads exactly the same value at that instant — in practice this
never happens with real recordings, but the handful of trials (if any) where it does are
dropped before the log transform rather than producing `inf`/`NaN` rows.

Coefficients are on the log scale: `exp(coef)` is a **fold change relative to baseline**
— e.g. `exp(coef) = 2` means the modeled condition's GFP is, on average, twice its own
pre-stimulus baseline, holding the other factors in the model fixed. This complements
Section 7's absolute-amplitude (µV) result with a per-trial-normalized, unit-free view of
the same intensity × frequency structure.

Output: `outputs/stats/sub-<ID>_ref-<ref>_lmm_ratio_GFP.tsv` +
combined `all_subjects_lmm_ratio_GFP.tsv` (per-coefficient), and
`sub-<ID>_ref-<ref>_lmm_omnibus_ratio_GFP.tsv` +
combined `all_subjects_lmm_omnibus_ratio_GFP.tsv` (Type III omnibus)

Same columns as Section 7/7b, plus `n_dropped_zero_baseline` printed per fit.


In [ ]:
lmm_ratio_rows = []
omnibus_ratio_rows = []

_, baseline_tmin, baseline_tmax = BASELINE_WINDOW   # from Section 7c: (-100.0, 0.0)

for ref_label, (pipeline, gfp_exclude, amp_exclude) in SEC7_REFERENCES.items():
    print(f'\n=== reference: {ref_label} ===')

    for subject in SUBJECTS:
        print(f'\n  sub-{subject}')
        epochs_by_intensity = load_epochs_for_subject(
            BIDS_ROOT, subject, pipeline, INTENSITIES
        )
        if len(epochs_by_intensity) < 2:
            print(f'  sub-{subject}: fewer than 2 intensity levels loaded — skipping LMM')
            continue

        ref_epo   = next(iter(epochs_by_intensity.values()))
        all_picks = get_all_eeg_picks(ref_epo)

        for comp_name, polarity, search_tmin, search_tmax in COMPONENTS:
            w0, w1 = analysis_windows.get((subject, ref_label, comp_name), (np.nan, np.nan))
            if np.isnan(w0):
                print(f'    {comp_name}: no analysis window — skipping')
                continue

            # Same trial set as Section 7, but each trial's amplitude is expressed
            # as a ratio to that trial's own pre-stimulus baseline GFP before
            # taking logs — mean_gfp_over_window is called twice on the same
            # cond_epo (component window, then baseline window), so trials line
            # up 1:1 by construction (both calls iterate the same Epochs object).
            rows = []
            n_dropped = 0
            for intensity, epo in epochs_by_intensity.items():
                for freq in FREQUENCIES:
                    cond_label = f'{intensity}{freq}'
                    if cond_label not in epo.event_id:
                        continue
                    cond_epo = epo[cond_label]
                    if len(cond_epo) == 0:
                        continue
                    amps      = mean_gfp_over_window(cond_epo, all_picks, w0, w1)
                    baselines = mean_gfp_over_window(cond_epo, all_picks, baseline_tmin, baseline_tmax)
                    cycle     = cond_epo.metadata['run'].to_numpy()
                    for amp, base, cyc in zip(amps, baselines, cycle):
                        if base <= 0:
                            n_dropped += 1
                            continue
                        rows.append(dict(
                            intensity=intensity, frequency=freq,
                            run_id=f'{intensity}_run{cyc}',
                            log_ratio=float(np.log(amp / base)),
                        ))

            if not rows:
                print(f'    {comp_name}: no data — skipping')
                continue

            long_df = pd.DataFrame(rows)

            try:
                model = smf.mixedlm(
                    'log_ratio ~ C(intensity) * C(frequency)',
                    data=long_df,
                    groups=long_df['run_id'],
                )
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter('always')
                    fit = model.fit(reml=True, method='lbfgs')
                boundary = any(
                    'singular' in str(w.message) or 'boundary' in str(w.message)
                    for w in caught
                )

                for term, coef in fit.params.items():
                    if term == 'Group Var':
                        continue
                    lmm_ratio_rows.append(dict(
                        reference=ref_label,
                        subject=subject,
                        component=comp_name,
                        term=term,
                        coef=round(float(coef), 6),
                        std_err=round(float(fit.bse[term]), 6),
                        z=round(float(fit.tvalues[term]), 4),
                        p=round(float(fit.pvalues[term]), 6),
                        ci_lower=round(float(fit.conf_int().loc[term, 0]), 6),
                        ci_upper=round(float(fit.conf_int().loc[term, 1]), 6),
                    ))

                wald = fit.wald_test_terms(scalar=True)
                for term_name in ['C(intensity)', 'C(frequency)', 'C(intensity):C(frequency)']:
                    row = wald.table.loc[term_name]
                    unreliable = boundary and term_name == 'C(intensity)'
                    omnibus_ratio_rows.append(dict(
                        reference=ref_label,
                        subject=subject,
                        component=comp_name,
                        term=term_name,
                        chi2=round(float(row['statistic']), 4),
                        df=int(row['df_constraint']),
                        p=round(float(row['pvalue']), 6),
                        unreliable_at_boundary=unreliable,
                    ))

                group_var = float(fit.params['Group Var'])
                n_sig = int((fit.pvalues.drop('Group Var', errors='ignore') < 0.05).sum())
                flag = '  [boundary: Group Var≈0 — C(intensity) omnibus unreliable]' if boundary else ''
                dropped_note = f'  [{n_dropped} trials dropped: baseline GFP <= 0]' if n_dropped else ''
                print(f'    {comp_name}: converged={fit.converged}  '
                      f'Group Var={group_var:.6g}  '
                      f'{n_sig}/{len(fit.params) - 1} terms p<.05{flag}{dropped_note}')
            except Exception as e:
                print(f'    {comp_name}: LMM failed — {e}')

lmm_ratio_df = pd.DataFrame(lmm_ratio_rows)
print(f'\nTotal ratio LMM result rows: {len(lmm_ratio_df)}')

omnibus_ratio_df = pd.DataFrame(omnibus_ratio_rows)
print(f'Total ratio omnibus (Type III Wald) result rows: {len(omnibus_ratio_df)}')

# Per-subject per-reference + combined files (same pattern as Section 7b)
for ref_label in lmm_ratio_df['reference'].unique():
    for subject in lmm_ratio_df['subject'].unique():
        subset = lmm_ratio_df[(lmm_ratio_df['reference'] == ref_label) & (lmm_ratio_df['subject'] == subject)]
        if subset.empty:
            continue
        fname = OUTPUT_DIR / f'sub-{subject}_ref-{ref_label}_lmm_ratio_GFP.tsv'
        subset.to_csv(fname, sep='\t', index=False)
        print(f'Saved: {fname.relative_to(REPO_ROOT)}  ({len(subset)} rows)')

combined_path = OUTPUT_DIR / 'all_subjects_lmm_ratio_GFP.tsv'
lmm_ratio_df.to_csv(combined_path, sep='\t', index=False)
print(f'Saved: {combined_path.relative_to(REPO_ROOT)}  (combined, {len(lmm_ratio_df)} rows)')

for ref_label in omnibus_ratio_df['reference'].unique():
    for subject in omnibus_ratio_df['subject'].unique():
        subset = omnibus_ratio_df[(omnibus_ratio_df['reference'] == ref_label) & (omnibus_ratio_df['subject'] == subject)]
        if subset.empty:
            continue
        fname = OUTPUT_DIR / f'sub-{subject}_ref-{ref_label}_lmm_omnibus_ratio_GFP.tsv'
        subset.to_csv(fname, sep='\t', index=False)
        print(f'Saved: {fname.relative_to(REPO_ROOT)}  ({len(subset)} rows)')

omnibus_combined_path = OUTPUT_DIR / 'all_subjects_lmm_omnibus_ratio_GFP.tsv'
omnibus_ratio_df.to_csv(omnibus_combined_path, sep='\t', index=False)
print(f'Saved: {omnibus_combined_path.relative_to(REPO_ROOT)}  (combined, {len(omnibus_ratio_df)} rows)')

lmm_ratio_df.head(15)


---

## 8. Simple-effects tests (intensity × frequency)

Section 7's `(1 | run_id)` random intercept works fine for the omnibus
intensity×frequency interaction, but for a *single-factor* simple-effects test the
between-run variance component can be estimated at exactly 0 for some
component/subject/reference combinations (a valid result — see Section 7 markdown) — and at
that exact boundary, statsmodels' `MixedLM` computes both the Wald standard errors and the
log-likelihood as `NaN`/`inf` (a `log(0)` term in the covariance determinant), not just a
noisy-but-usable estimate. That breaks both Wald tests and likelihood-ratio tests for these
simple effects, so Section 8/8b/8c use fixed-effects-only OLS instead of a random intercept —
`cycle` (the recording block, 1–4) is entered as a categorical **fixed** blocking factor
rather than a random intercept, matching how the study design actually replicates:

- **Frequency simple effect at each intensity** — trial-level:
  `amplitude ~ C(frequency) + C(cycle)` fit by OLS on the trials at that intensity
  (4 cycles/blocks). Tests whether frequency has an effect on GFP within one intensity,
  controlling for which of the 4 recording blocks a trial came from.
- **Intensity simple effect at each frequency** — run-level: intensity only varies
  *between* runs (each run is entirely one intensity), so the effective replication for an
  intensity effect is 4 cycles per intensity, not the trial count. Trials at that frequency
  are averaged within each of the 12 runs first (`GFP_run_mean = mean(amplitude)` over the
  trials of that run/frequency), giving 12 observations (3 intensities × 4 cycles), then
  `GFP_run_mean ~ C(intensity) + C(cycle)` is fit by OLS on those 12 run-level means.

Both use ordinary OLS (Type II ANOVA F-test via `anova_lm`) — no random effect, so no
boundary-variance failure mode is possible.

`*` in the printed summary marks p < 0.05.  
The full long-form data (with `run_id`/`cycle`) is stored in `sec10_balanced` and reused by
**Section 8b/8c** (next cells) for pairwise comparisons, at the same trial-level /
run-level granularity as above.

Output: `outputs/stats/sub-<ID>_ref-<ref>_simple_effects_GFP.tsv`

In [ ]:
# ── Section 8 settings ──────────────────────────────────────────────────────
SEC10_AMPLITUDE = 'GFP'   # 'GFP' only; per-channel not implemented here
SEC10_PADJUST   = 'bonf'  # p-adjustment used in Section 8b/8c pairwise tests
# ─────────────────────────────────────────────────────────────────────────────

from statsmodels.stats.anova import anova_lm

# Map pingouin-style padjust names (used by SEC10_PADJUST) to
# statsmodels.stats.multitest.multipletests method names, used in Section 8b/8c.
_PADJUST_TO_MULTIPLETESTS_METHOD = {
    'bonf': 'bonferroni', 'holm': 'holm', 'fdr_bh': 'fdr_bh', 'fdr_by': 'fdr_by',
}

_freq_lbl = {}
for _f in FREQUENCIES:
    _hz = int(_f.replace('Hz', ''))
    _freq_lbl[_f] = f'{_hz // 1000}k' if _hz >= 1000 else str(_hz)


def anova_term_p(ols_fit, term_prefix):
    """Type II ANOVA F-test p-value for one term of an OLS fit (statsmodels anova_lm)."""
    table = anova_lm(ols_fit, typ=2)
    matches = [r for r in table.index if r.startswith(term_prefix)]
    if len(matches) != 1:
        raise KeyError(f'expected exactly one anova_lm row starting with {term_prefix!r}, got {matches}')
    row = table.loc[matches[0]]
    return float(row['PR(>F)']), float(row['F']), float(row['df'])


sec10_rows     = []   # simple-effect OLS results
sec10_balanced = {}   # (ref_label, subject, comp_name) → trial-level long DataFrame (incl. run_id, cycle)

# Section 8 (and 8c) analyze linked-ears data only.
SEC8_REFERENCES = {'linkedEars': REFERENCES['linkedEars']}

for ref_label, (pipeline, gfp_exclude, amp_exclude) in SEC8_REFERENCES.items():
    print(f'\n=== reference: {ref_label} ===')

    for subject in SUBJECTS:
        print(f'\n  sub-{subject}')
        epochs_by_intensity = load_epochs_for_subject(
            BIDS_ROOT, subject, pipeline, INTENSITIES
        )
        if len(epochs_by_intensity) < 2:
            print(f'  sub-{subject}: fewer than 2 intensity levels — skipping')
            continue

        ref_epo      = next(iter(epochs_by_intensity.values()))
        picks_for_dv = get_all_eeg_picks(ref_epo)

        for comp_name, polarity, search_tmin, search_tmax in COMPONENTS:
            w0, w1 = analysis_windows.get((subject, ref_label, comp_name), (np.nan, np.nan))
            if np.isnan(w0):
                continue

            rows = []
            for intensity, epo in epochs_by_intensity.items():
                for freq in FREQUENCIES:
                    cond_label = f'{intensity}{freq}'
                    if cond_label not in epo.event_id:
                        continue
                    cond_epo = epo[cond_label]
                    if len(cond_epo) == 0:
                        continue
                    amps  = mean_gfp_over_window(cond_epo, picks_for_dv, w0, w1)
                    cycle = cond_epo.metadata['run'].to_numpy()
                    for amp, cyc in zip(amps, cycle):
                        rows.append(dict(
                            intensity=intensity, frequency=freq,
                            cycle=str(cyc), run_id=f'{intensity}_run{cyc}',
                            amplitude=amp,
                        ))

            if not rows:
                continue

            long_df = pd.DataFrame(rows)
            sec10_balanced[(ref_label, subject, comp_name)] = long_df

            print(f'    {comp_name}:')

            # ── Frequency simple effect at each intensity — trial-level OLS ───
            # amplitude ~ C(frequency) + C(cycle), 4 cycles/blocks per intensity.
            int_marks = []
            for intensity in INTENSITIES:
                subset = long_df[long_df['intensity'] == intensity]
                try:
                    ols_fit = smf.ols('amplitude ~ C(frequency) + C(cycle)', data=subset).fit()
                    p, F, df_num = anova_term_p(ols_fit, 'C(frequency)')
                    sec10_rows.append(dict(
                        reference=ref_label, subject=subject, component=comp_name,
                        test_factor='frequency', fixed_factor='intensity', fixed_level=intensity,
                        granularity='trial', F=round(F, 4), df_num=df_num, p=round(p, 6),
                    ))
                    int_marks.append(intensity + ('*' if p < 0.05 else ''))
                except Exception as e:
                    print(f'      frequency@{intensity}: failed — {e}')
                    int_marks.append(intensity + '!')
            print(f'      frequency @ intensity: {" ".join(int_marks)}')

            # ── Intensity simple effect at each frequency — run-level OLS ─────
            # Average trials to one GFP value per run first (12 runs = 3 intensities
            # x 4 cycles), since intensity only varies between runs, not within one.
            freq_marks = []
            run_means = (
                long_df.groupby(['intensity', 'cycle', 'frequency'])['amplitude']
                .mean().reset_index().rename(columns={'amplitude': 'gfp_run_mean'})
            )
            for freq in FREQUENCIES:
                subset = run_means[run_means['frequency'] == freq]
                try:
                    ols_fit = smf.ols('gfp_run_mean ~ C(intensity) + C(cycle)', data=subset).fit()
                    p, F, df_num = anova_term_p(ols_fit, 'C(intensity)')
                    sec10_rows.append(dict(
                        reference=ref_label, subject=subject, component=comp_name,
                        test_factor='intensity', fixed_factor='frequency', fixed_level=freq,
                        granularity='run', F=round(F, 4), df_num=df_num, p=round(p, 6),
                    ))
                    freq_marks.append(_freq_lbl[freq] + ('*' if p < 0.05 else ''))
                except Exception as e:
                    print(f'      intensity@{freq}: failed — {e}')
                    freq_marks.append(_freq_lbl[freq] + '!')
            print(f'      intensity @ freq:     {" ".join(freq_marks)}')

sec10_df = pd.DataFrame(sec10_rows)
print(f'\nSimple-effect rows: {len(sec10_df)}')
if len(sec10_df):
    display(sec10_df.head(10))

# ── Save ─────────────────────────────────────────────────────────────────────
if not sec10_df.empty:
    for ref_label in sec10_df['reference'].unique():
        for subject in sec10_df['subject'].unique():
            subset = sec10_df[
                (sec10_df['reference'] == ref_label) & (sec10_df['subject'] == subject)
            ]
            if subset.empty:
                continue
            fname = OUTPUT_DIR / f'sub-{subject}_ref-{ref_label}_simple_effects_GFP.tsv'
            subset.to_csv(fname, sep='\t', index=False)
            print(f'Saved: {fname.relative_to(REPO_ROOT)}  ({len(subset)} rows)')
    combined_path = OUTPUT_DIR / 'all_subjects_simple_effects_GFP.tsv'
    sec10_df.to_csv(combined_path, sep='\t', index=False)
    print(f'Saved: {combined_path.relative_to(REPO_ROOT)}  (combined, {len(sec10_df)} rows)')

In [ ]:
## 8b. Pairwise frequency comparisons at each intensity level (OLS)
# Reuses `sec10_balanced` built above — no epoch re-loading needed.
# For each intensity, tests all pairs of the 8 frequency conditions (28 pairs × 3 intensities)
# with trial-level OLS: amplitude ~ C(frequency_pair) + C(cycle), same design as Section 8's
# frequency simple effect (fixed cycle block, no random effect — see Section 8 markdown for
# why a random intercept is avoided here).
# ─────────────────────────────────────────────────────────────────────────────

from itertools import combinations
from statsmodels.stats.multitest import multipletests

sec10b_rows = []

# Section 8b analyzes linked-ears data only.
SEC8B_REFERENCES = {'linkedEars': REFERENCES['linkedEars']}

for ref_label, (pipeline, *_) in SEC8B_REFERENCES.items():
    print(f'\n=== reference: {ref_label} ===')

    for subject in SUBJECTS:
        print(f'\n  sub-{subject}')
        for comp_name, *_ in COMPONENTS:
            key = (ref_label, subject, comp_name)
            if key not in sec10_balanced:
                continue
            balanced = sec10_balanced[key]

            print(f'    {comp_name}:')
            for intensity in INTENSITIES:
                subset = balanced[balanced['intensity'] == intensity]
                pair_rows = []
                for freq_a, freq_b in combinations(FREQUENCIES, 2):
                    pair_df = subset[subset['frequency'].isin([freq_a, freq_b])]
                    try:
                        fit = smf.ols('amplitude ~ C(frequency) + C(cycle)', data=pair_df).fit()
                        term = [t for t in fit.params.index if t.startswith('C(frequency)')][0]
                        pair_rows.append(dict(
                            A=freq_a, B=freq_b,
                            coef=float(fit.params[term]),
                            t=float(fit.tvalues[term]),
                            p_unc=float(fit.pvalues[term]),
                        ))
                    except Exception as e:
                        print(f'      {intensity} {freq_a} vs {freq_b}: failed — {e}')

                if not pair_rows:
                    continue
                p_uncs   = [r['p_unc'] for r in pair_rows]
                p_adjust = multipletests(p_uncs, method=_PADJUST_TO_MULTIPLETESTS_METHOD[SEC10_PADJUST])[1]
                for r, p_adj in zip(pair_rows, p_adjust):
                    sec10b_rows.append(dict(
                        reference=ref_label, subject=subject, component=comp_name,
                        intensity=intensity,
                        padjust=SEC10_PADJUST,
                        A=r['A'], B=r['B'],
                        coef=round(r['coef'], 6), t=round(r['t'], 4),
                        p_unc=round(r['p_unc'], 6), p_adj=round(float(p_adj), 6),
                    ))

                n_sig, n_pairs = int((p_adjust < 0.05).sum()), len(pair_rows)
                print(f'      {intensity}: {n_sig}/{n_pairs} pairs p_adj<.05')

sec10b_df = pd.DataFrame(sec10b_rows)
print(f'\nPairwise frequency rows: {len(sec10b_df)}')
if len(sec10b_df):
    display(sec10b_df.head(10))

# ── 1000 Hz comparisons ───────────────────────────────────────────────────
# `sec10b_df`'s coef/t are always (amplitude[B] - amplitude[A]) from the OLS fit,
# where A/B are the two frequencies in ascending order (see `combinations` above)
# — NOT consistently anchored to 1000 Hz. For frequencies below 1000 Hz, 1000 Hz
# is B, so coef = amp[1000Hz] - amp[freq]; for frequencies above 1000 Hz, 1000 Hz
# is A, so coef = amp[freq] - amp[1000Hz] — the opposite sign convention. Left
# as-is this makes the raw coef/t sign flip depending on which side of 1000 Hz a
# row is on. Re-sign here so coef/t consistently mean (1000 Hz - frequency),
# collapsing A/B into a single `frequency` column (the non-1000-Hz partner).
if len(sec10b_df):
    mask_1k = (sec10b_df['A'] == '01000Hz') | (sec10b_df['B'] == '01000Hz')
    df_1k = sec10b_df[mask_1k].copy().reset_index(drop=True)

    _sign = np.where(df_1k['B'] == '01000Hz', 1.0, -1.0)
    df_1k['frequency'] = np.where(df_1k['B'] == '01000Hz', df_1k['A'], df_1k['B'])
    df_1k['coef'] = df_1k['coef'] * _sign
    df_1k['t']    = df_1k['t'] * _sign
    df_1k = df_1k.drop(columns=['A', 'B'])
    df_1k = df_1k[['reference', 'subject', 'component', 'intensity', 'padjust',
                   'frequency', 'coef', 't', 'p_unc', 'p_adj']]

    out_1k = OUTPUT_DIR / 'all_subjects_pairwise_freq_GFP_1000Hz.tsv'
    df_1k.to_csv(out_1k, sep='\t', index=False)
    print(f'Saved 1000 Hz pairs ({len(df_1k)} rows): {out_1k.relative_to(REPO_ROOT)}')
    print('  coef/t are signed as (1000 Hz - frequency): positive = larger at 1000 Hz')
    display(df_1k)

# ── Save ──────────────────────────────────────────────────────────────────────
for label, out_df, tag in [
    ('pairwise frequency',  sec10b_df, 'pairwise_freq_GFP'),
]:
    if out_df.empty:
        print(f'No {label} results to save.')
        continue
    for ref_label in out_df['reference'].unique():
        for subject in out_df['subject'].unique():
            subset = out_df[
                (out_df['reference'] == ref_label) & (out_df['subject'] == subject)
            ]
            if subset.empty:
                continue
            fname = OUTPUT_DIR / f'sub-{subject}_ref-{ref_label}_{tag}.tsv'
            subset.to_csv(fname, sep='\t', index=False)
            print(f'Saved: {fname.relative_to(REPO_ROOT)}  ({len(subset)} rows)')
    combined_path = OUTPUT_DIR / f'all_subjects_{tag}.tsv'
    out_df.to_csv(combined_path, sep='\t', index=False)
    print(f'Saved: {combined_path.relative_to(REPO_ROOT)}  (combined, {len(out_df)} rows)')

In [ ]:
## 8c. Pairwise intensity comparisons at each frequency level (OLS, run-level)
# Reuses `sec10_balanced` built above — no epoch re-loading needed.
# For each frequency, tests all pairs of the 3 intensity conditions (3 pairs × 8 frequencies).
# Same run-level design as Section 8's intensity simple effect: intensity only varies
# between runs, so trials are averaged to one GFP value per run first (4 cycles per
# intensity), giving 8 observations per pair (2 intensities x 4 cycles), then
# gfp_run_mean ~ C(intensity_pair) + C(cycle) is fit by OLS (fixed cycle block, no random
# effect — see Section 8 markdown for why a random intercept is avoided here).
# ─────────────────────────────────────────────────────────────────────────────

from itertools import combinations
from statsmodels.stats.multitest import multipletests

sec10c_rows = []

for ref_label, (pipeline, *_) in SEC8_REFERENCES.items():
    print(f'=== reference: {ref_label} ===')

    for subject in SUBJECTS:
        print(f'sub-{subject}')
        for comp_name, *_ in COMPONENTS:
            key = (ref_label, subject, comp_name)
            if key not in sec10_balanced:
                continue
            balanced = sec10_balanced[key]
            run_means = (
                balanced.groupby(['intensity', 'cycle', 'frequency'])['amplitude']
                .mean().reset_index().rename(columns={'amplitude': 'gfp_run_mean'})
            )

            print(f'    {comp_name}:')
            for freq in FREQUENCIES:
                subset = run_means[run_means['frequency'] == freq]
                pair_rows = []
                for int_a, int_b in combinations(INTENSITIES, 2):
                    pair_df = subset[subset['intensity'].isin([int_a, int_b])]
                    try:
                        fit = smf.ols('gfp_run_mean ~ C(intensity) + C(cycle)', data=pair_df).fit()
                        term = [t for t in fit.params.index if t.startswith('C(intensity)')][0]
                        pair_rows.append(dict(
                            A=int_a, B=int_b,
                            coef=float(fit.params[term]),
                            t=float(fit.tvalues[term]),
                            p_unc=float(fit.pvalues[term]),
                        ))
                    except Exception as e:
                        print(f'      {freq} {int_a} vs {int_b}: failed — {e}')

                if not pair_rows:
                    continue
                p_uncs   = [r['p_unc'] for r in pair_rows]
                p_adjust = multipletests(p_uncs, method=_PADJUST_TO_MULTIPLETESTS_METHOD[SEC10_PADJUST])[1]
                for r, p_adj in zip(pair_rows, p_adjust):
                    sec10c_rows.append(dict(
                        reference=ref_label, subject=subject, component=comp_name,
                        frequency=freq,
                        padjust=SEC10_PADJUST,
                        A=r['A'], B=r['B'],
                        coef=round(r['coef'], 6), t=round(r['t'], 4),
                        p_unc=round(r['p_unc'], 6), p_adj=round(float(p_adj), 6),
                    ))

                n_sig, n_pairs = int((p_adjust < 0.05).sum()), len(pair_rows)
                print(f'      {freq}: {n_sig}/{n_pairs} pairs p_adj<.05')

sec10c_df = pd.DataFrame(sec10c_rows)
print(f'Pairwise intensity rows: {len(sec10c_df)}')
if len(sec10c_df):
    display(sec10c_df.head(10))

# ── Save ──────────────────────────────────────────────────────────────────────
for label, out_df, tag in [
    ('pairwise intensity', sec10c_df, 'pairwise_intensity_GFP'),
]:
    if out_df.empty:
        print(f'No {label} results to save.')
        continue
    for ref_label in out_df['reference'].unique():
        for subject in out_df['subject'].unique():
            subset = out_df[
                (out_df['reference'] == ref_label) & (out_df['subject'] == subject)
            ]
            if subset.empty:
                continue
            fname = OUTPUT_DIR / f'sub-{subject}_ref-{ref_label}_{tag}.tsv'
            subset.to_csv(fname, sep='\t', index=False)
            print(f'Saved: {fname.relative_to(REPO_ROOT)}  ({len(subset)} rows)')
    combined_path = OUTPUT_DIR / f'all_subjects_{tag}.tsv'
    out_df.to_csv(combined_path, sep='\t', index=False)
    print(f'Saved: {combined_path.relative_to(REPO_ROOT)}  (combined, {len(out_df)} rows)')

---

## 9. Amplitude vs frequency by intensity level

For each subject, one figure with 5 subplots (one per component), plotted for the
linked-ears reference only.  
X-axis: tone frequency (log scale, 125–16 000 Hz).  
Y-axis: GFP amplitude (or peak-channel ERP amplitude) at the fixed peak latency —
**the same fixed range is used for every component subplot and every
subject's figure**, computed once from the min/max amplitude across all of
them, so panels stay visually comparable.  
Lines: one per intensity level (45 dB, 60 dB, 75 dB).

**Amplitude is computed from each condition's AEP (evoked, i.e. trial-averaged), not from
individual trials**: AEP → GFP (or peak-channel signal) → mean over the analysis window,
giving exactly one value per (intensity, frequency, component) cell. Since there is only
one AEP per condition, there is no trial-level spread to show, so **no CI band is drawn** —
these are point estimates only, not visual proxies for a statistical test. For significance,
use Section 7 (omnibus intensity×frequency LMM) or Section 8/8b/8c (simple effects and
pairwise comparisons), which work from the underlying per-trial/per-run data.

Peak latencies are taken from Section 3 (estimated from the grand-average GFP).  
Set `PLOT11_AMPLITUDE` to `'GFP'` or `'peak_channel'`.


In [ ]:
# ── Section 9 settings ──────────────────────────────────────────────────────
PLOT11_AMPLITUDE = 'GFP'   # 'GFP' → GFP amplitude; 'peak_channel' → fixed peak-channel ERP
SAVE_DIR = '/workspace/figures'
# ─────────────────────────────────────────────────────────────────────────────

import ipywidgets as _w
from IPython.display import display as _display

def amplitude_over_window(data_uv, tmin_ms, tmax_ms, times_ms):
    """Mean amplitude (µV) over [tmin_ms, tmax_ms] for a single evoked's data.
    data_uv shape: (n_ch, n_times) — pass GFP as a (1, n_times) row, or a
    single/multi-channel signal slice, to get one scalar amplitude back.
    """
    mask = (times_ms >= tmin_ms) & (times_ms <= tmax_ms)
    if not mask.any():
        return np.nan
    return float(data_uv[:, mask].mean())

freq_hz = [int(f.replace('Hz', '')) for f in FREQUENCIES]
intensity_colors = {'45dB': '#2196F3', '60dB': '#4EB458', '75dB': '#F44336'}
y_label = 'GFP amplitude (µV)' if PLOT11_AMPLITUDE == 'GFP' else 'ERP amplitude (µV)'

# ── Pass 1: compute amplitudes for every (reference, subject) so the Y-axis
# range below can be fixed across all figures, not just within one figure ────
# Section 9 plots linked-ears data only.
SEC9_REFERENCES = {'linkedEars': REFERENCES['linkedEars']}

_panels = []
for ref_label, (pipeline, gfp_exclude, amp_exclude) in SEC9_REFERENCES.items():
    for subject in SUBJECTS:

        evokeds_by_cond = load_evokeds_for_subject(
            BIDS_ROOT, subject, pipeline, INTENSITIES, FREQUENCIES
        )
        if not evokeds_by_cond:
            print(f'sub-{subject} ref-{ref_label}: no evoked files — skipping')
            continue

        ref_evoked = next(iter(evokeds_by_cond.values()))
        all_picks  = get_all_eeg_picks(ref_evoked, gfp_exclude)
        sig_picks  = get_signal_picks(ref_evoked, amp_exclude)
        times_ms   = ref_evoked.times * 1000

        # Peak channel per component (from the 75 dB evoked, averaged
        # equally across the 8 frequencies)
        evokeds_75  = {
            cond: ev for cond, ev in evokeds_by_cond.items()
            if cond.startswith(LATENCY_INTENSITY)
        }
        sig_mean_75 = np.mean(
            [ev.data[sig_picks] * 1e6 for ev in evokeds_75.values()], axis=0
        ) if evokeds_75 else None

        peak_ch = {}
        for comp_name, *_ in COMPONENTS:
            lat = peak_latencies.get((subject, ref_label, comp_name), np.nan)
            if np.isnan(lat) or sig_mean_75 is None:
                peak_ch[comp_name] = None
                continue
            t_idx  = int(np.argmin(np.abs(times_ms - lat)))
            ch_idx = int(np.argmax(np.abs(sig_mean_75[:, t_idx])))
            peak_ch[comp_name] = ref_evoked.ch_names[sig_picks[ch_idx]]

        # Amplitude per (intensity, freq, component) from the condition's
        # AEP (evoked): AEP → GFP (or peak-channel signal) → window mean.
        # One value per cell — no trial-level spread, so no CI band.
        amp_by_cell = {}
        for cond, ev in evokeds_by_cond.items():
            intensity = next((i for i in INTENSITIES if cond.startswith(i)), None)
            freq      = cond[len(intensity):] if intensity else None
            if intensity is None or freq not in FREQUENCIES:
                continue

            for comp_name, *_ in COMPONENTS:
                w0, w1 = analysis_windows.get(
                    (subject, ref_label, comp_name), (np.nan, np.nan)
                )
                if np.isnan(w0):
                    continue

                if PLOT11_AMPLITUDE == 'GFP':
                    gfp = compute_gfp(ev.data[all_picks] * 1e6)[np.newaxis, :]
                    amp = amplitude_over_window(gfp, w0, w1, times_ms)
                else:
                    ch = peak_ch.get(comp_name)
                    if ch is None:
                        continue
                    ch_pick = [ref_evoked.ch_names.index(ch)]
                    amp = amplitude_over_window(ev.data[ch_pick] * 1e6, w0, w1, times_ms)

                amp_by_cell[(intensity, freq, comp_name)] = amp

        _panels.append(dict(
            ref_label=ref_label, subject=subject,
            amp_by_cell=amp_by_cell, peak_ch=peak_ch,
        ))

# Fixed Y-axis range shared by every figure and every component subplot —
# computed once from all amplitudes across all subjects/references so panels
# stay visually comparable.
_all_amps = [v for p in _panels for v in p['amp_by_cell'].values() if not np.isnan(v)]
_y_min, _y_max = min(_all_amps), max(_all_amps)
_y_pad = 0.05 * (_y_max - _y_min) if _y_max > _y_min else 1.0
Y_RANGE = (_y_min - _y_pad, _y_max + _y_pad)

# ── Pass 2: draw one figure per (reference, subject), all sharing Y_RANGE ────
for panel in _panels:
    ref_label   = panel['ref_label']
    subject     = panel['subject']
    amp_by_cell = panel['amp_by_cell']
    peak_ch     = panel['peak_ch']

    n_comp = len(COMPONENTS)
    fig, axes = plt.subplots(1, n_comp, figsize=(3.2 * n_comp, 4), sharey=True)
    fig.suptitle(
        f'sub-{subject}  ref-{ref_label}  —  {y_label}\n'
        f'(mean over analysis window, computed from each condition\'s AEP — '
        f'see Section 7/8 for run-clustered significance tests)',
        fontsize=9, y=1.02,
    )

    for ax, (comp_name, polarity, *_) in zip(axes, COMPONENTS):
        for intensity in INTENSITIES:
            col = intensity_colors[intensity]
            means = [
                amp_by_cell.get((intensity, freq, comp_name), np.nan)
                for freq in FREQUENCIES
            ]
            ax.plot(freq_hz, means,
                    color=col, marker='o', markersize=4, lw=1.5, label=intensity)

        ax.set_ylim(*Y_RANGE)
        ax.set_xscale('log')
        ax.set_xticks(freq_hz)
        ax.set_xticklabels(['125', '250', '500', '1k', '2k', '4k', '8k', '16k'],
                           rotation=45, ha='right', fontsize=7)

        ch = peak_ch.get(comp_name)
        if PLOT11_AMPLITUDE == 'peak_channel' and ch:
            ax.set_title(f'{comp_name} ({ch})', fontsize=10)
        else:
            ax.set_title(comp_name, fontsize=10)

        if PLOT11_AMPLITUDE == 'peak_channel' and polarity == -1:
            ax.invert_yaxis()

        ax.set_xlabel('Frequency (Hz)', fontsize=8)
        ax.grid(True, which='both', alpha=0.3, lw=0.5)

    axes[0].set_ylabel(y_label, fontsize=8)
    axes[-1].legend(title='Intensity', fontsize=8, title_fontsize=8)
    fig.tight_layout()
    plt.show()

    _fig_s  = fig
    _name_s = _w.Text(value=f'sub-{subject}_ref-{ref_label}_GFP-vs-freq',
                      description='Filename:', layout=_w.Layout(width='400px'))
    _fmt_s  = _w.Dropdown(options=['pdf', 'svg', 'png'], value='pdf',
                          description='Format:', layout=_w.Layout(width='160px'))
    _btn_s  = _w.Button(description='Save', button_style='primary',
                        layout=_w.Layout(width='80px'))
    _out_s  = _w.Output()

    def _save_s(_, f=_fig_s, n=_name_s, fmt=_fmt_s, o=_out_s):
        with o:
            o.clear_output()
            path = f'{SAVE_DIR}/{n.value}.{fmt.value}'
            f.savefig(path, format=fmt.value, bbox_inches='tight')
            print(f'Saved: {path}')

    _btn_s.on_click(_save_s)
    _display(_w.HBox([_name_s, _fmt_s, _btn_s]), _out_s)
